# FINA4030A — Lab 6
## Auditing work that is mostly right

**Class 6.** Submit this notebook by 23:59 on **29 October**.

> **Before you type anything: File → Save a copy in Drive.**

You have a mean-variance optimiser on the Ken French 49 industry portfolios,
built by an agent, and the request that produced it. Read the request first.

**This lab was going to be a different lab.** It was designed around a failure
that did not happen. Asked to optimise on thirty-six months of data across
forty-nine assets — a covariance matrix that cannot be inverted — the system was
expected to invert it anyway, produce a confident efficient frontier, and say
nothing. Instead it reported the rank, named the fourteen portfolio directions
with zero estimated risk, applied shrinkage without being asked, refused to
invert, and led its own summary with evidence that at that window the optimised
portfolios lose to doing nothing at all.

So the naive failure is gone, and you are auditing something much more like what
you will actually be handed: **work that is careful, well documented, largely
correct, and wrong in one place that matters.**

**Four passes.**

1. **Verify what it asserts.** Most of it is true. Establish which.
2. **Find what it never tested.** One conclusion rests on a comparison that does
   not do what it claims. Its own output contains the proof.
3. **Re-run it properly** and measure how much the conclusion changes.
4. **Say what the verdict is.** "Wrong" and "overstated" are different findings
   and attract different responses.

**What is marked.** Passes 2 to 4, and the number you attach to the correction.
Confirming that the correct parts are correct is necessary and earns little on
its own — an auditor who only ever agrees is not adding anything.

**A warning about this specimen.** It is good. You will be tempted to accept it
because the parts you check keep coming back clean. That tempation *is* the
exercise: the failure mode of a competent reviewer is not missing an obvious
error, it is granting credibility earned in one place to a claim made in another.


In [ ]:
# Setup. Run this first.
REQUIRED_CLIENT = "1.1"
REPO = "https://raw.githubusercontent.com/fy-ericlam/fina4030a/main"

import importlib, sys, urllib.request, warnings
warnings.filterwarnings("ignore")

urllib.request.urlretrieve(f"{REPO}/fina4030a.py", "fina4030a.py")
for f in ("lab06_returns.json", "lab06_optimiser.py", "lab06_request.txt",
          "lab06_run_log.txt", "lab06_backtest.csv", "lab06_summary.csv",
          "lab06_window_sensitivity.csv"):
    urllib.request.urlretrieve(f"{REPO}/labs/{f}", f)

sys.modules.pop("fina4030a", None)
import fina4030a
importlib.reload(fina4030a)

if fina4030a.__version__ < REQUIRED_CLIENT:
    print(f"!! Loaded client v{fina4030a.__version__}, needs v{REQUIRED_CLIENT}.")
    print("   Runtime > Restart session, then run this cell again.")
else:
    print(f"client v{fina4030a.__version__} loaded")

# --- your details -----------------------------------------------------------
NAME       = ""
STUDENT_ID = ""

fina4030a.configure(provider="cuhk_portal")
fina4030a.verify()


---
## The request, and what came back

Everything the system was told is in the first cell below. Everything it was not
told is there too — nothing about conditioning, rank, shrinkage, or how many
observations thirty-six months buys you across forty-nine assets.


In [ ]:
print(open("lab06_request.txt", encoding="utf-8").read())


In [ ]:
# Its own run log, as delivered. Read it. This is the thing you are auditing.
print(open("lab06_run_log.txt", encoding="utf-8").read())


---

## Calibration log — the top half now, before you go any further

Fill the **before** fields in this cell now, while you still do not know the
answer. Being wrong here costs you nothing. Being wrong *and not noticing* is
the thing this course exists to train out of you, and systematic overconfidence
is a finding I mark as one.

Come back for the **after** fields at the end: edit the cell and run it again.


In [ ]:
CALIBRATION = {
    # --- Before Pass 0. You have read the request and its own run log. ---
    # The cover note draws a conclusion. Will it survive correction?
    "expected_survives_correction": None,   # True / False
    # How far would you trust this optimiser's output right now?
    "trust_before":                 None,   # 1 = not at all, 5 = would sign it
    # --- After. Straight from VERDICT, at Pass 4. ---
    "actual_survives_correction":   None,   # True / False
    "trust_after":                  None,   # 1-5, same scale
    "surprise":                     "",     # one sentence: what you did not expect
}

_todo = [k for k, v in CALIBRATION.items()
         if v is None or (isinstance(v, str) and not v.strip())]
print("Calibration log complete."
      if not _todo else "Still to fill in: " + ", ".join(_todo))


---
# Pass 0 — establish the ground yourself

Before checking anything it said, work out what the data can support. This takes
one cell and it is the fact the whole optimisation rests on.

With **N** assets you need more than **N** independent observations before a
sample covariance matrix can be inverted at all. Below that, some combination of
assets has *exactly zero* estimated variance — not small, zero — and the
optimiser will happily put all your money in it.


In [ ]:
import json, numpy as np, pandas as pd

d = json.load(open("lab06_returns.json", encoding="utf-8"))
R_pct = np.array(d["returns_pct"], float)
R = R_pct / 100.0
names, dates = d["industries"], d["dates"]
N = R.shape[1]
print(f"{R.shape[0]} months x {N} industries, {dates[0]} to {dates[-1]}")

rows = []
for w in (24, 36, 48, 49, 50, 60, 120, 240):
    S = np.cov(R[-w:], rowvar=False)
    rows.append({"window_months": w, "N": N,
                 "rank": np.linalg.matrix_rank(S),
                 "full_rank": np.linalg.matrix_rank(S) == N,
                 "condition_number": f"{np.linalg.cond(S):.3g}"})
print(pd.DataFrame(rows).to_string(index=False))


Look at where the cliff is, and note that it is not gradual.

**Now a second question, which matters more than it looks.** You just computed
two diagnostics of the same matrix. Are they equally trustworthy? The next cell
computes both on data that differs only in its units — percent against decimals
— which cannot change any real property of the matrix.


In [ ]:
print(f"{'window':>7}{'rank (pct)':>12}{'rank (dec)':>12}"
      f"{'cond (pct)':>14}{'cond (dec)':>14}{'ratio':>10}")
for w in (24, 36, 48, 60, 120, 240):
    Sp, Sd = np.cov(R_pct[-w:], rowvar=False), np.cov(R[-w:], rowvar=False)
    cp, cd = np.linalg.cond(Sp), np.linalg.cond(Sd)
    print(f"{w:>7}{np.linalg.matrix_rank(Sp):>12}{np.linalg.matrix_rank(Sd):>12}"
          f"{cp:>14.3g}{cd:>14.3g}{cp / cd:>10.2f}")


Scaling a matrix by 100 leaves its condition number mathematically unchanged.
Above the cliff the two columns agree exactly. Below it they do not, and the
disagreement is not small.

Once a matrix is numerically singular its smallest eigenvalue is floating-point
noise, so anything computed by dividing by it is noise too. **The condition
number of a singular matrix is not reproducible — across units, libraries or
machines. The rank is.**

This is worth carrying beyond today. A diagnostic that disagrees with itself
depending on how you stored the data is not a diagnostic.


---
# Pass 1 — verify what it asserts

The run log makes a series of specific, checkable claims. Check them. Most will
hold; you need to know which, because everything in pass 2 depends on knowing
where this thing is reliable.

`lab06_optimiser.py` is the delivered code. You can import it.


In [ ]:
import lab06_optimiser as OPT
from sklearn.covariance import ledoit_wolf as sklearn_lw

data = OPT.load_data("lab06_returns.json")
W36 = data["R"][-36:]
rf36 = float(np.nanmean(data["rf"][-36:]))
mu36 = W36.mean(axis=0)
S_raw = np.cov(W36, rowvar=False)
S_lw, k_lw = OPT.cov_ledoit_wolf(W36)

def claim(text, got, expected, tol=None):
    ok = (abs(got - expected) <= tol) if tol is not None else (got == expected)
    print(f"  [{'holds' if ok else 'FAILS':<5}] {text:<52} got {got}")
    return ok

print("claims made in the run log:\n")
claim("rank of the 36-month sample covariance is 35",
      int(np.linalg.matrix_rank(S_raw)), 35)
claim("14 eigenvalues are (numerically) zero",
      int((np.linalg.eigvalsh(S_raw) < 1e-14).sum()), 14)
claim("Ledoit-Wolf shrinkage intensity is 0.161",
      round(k_lw, 3), 0.161, tol=0.0005)
claim("condition number after shrinkage is 108.3",
      round(float(np.linalg.cond(S_lw)), 1), 108.3, tol=0.2)
claim("risk-free over the window is 4.58% annualised",
      round(rf36 * 12 * 100, 2), 4.58, tol=0.01)

# It claims its hand-rolled shrinkage matches the standard implementation.
S_sk, k_sk = sklearn_lw(W36, assume_centered=False)
print(f"\n  its Ledoit-Wolf vs sklearn's: max abs difference "
      f"{np.abs(S_lw - S_sk).max():.2e}, intensity {k_lw:.6f} vs {k_sk:.6f}")


Now the two portfolio results, which you should reproduce rather than accept.
This calls the delivered optimiser's own functions — so it checks that the
numbers in the tables came from the code, not that the code is right.


In [ ]:
w_mv = OPT.min_variance(mu36, S_lw, True)
w_tn = OPT.tangency(mu36, S_lw, rf36, True)
for nm, w in (("min_variance", w_mv), ("tangency", w_tn)):
    s = OPT.port_stats(w, mu36, S_lw, rf36)
    print(f"  {nm:<14} ann return {s['ann_return']*100:6.2f}%   "
          f"ann vol {s['ann_vol']*100:6.2f}%   Sharpe {s['sharpe']:.2f}")
print("\nsummary.csv as delivered:")
print(pd.read_csv("lab06_summary.csv").to_string(index=False))


In [ ]:
PASS1 = {
    "claims_checked":      None,   # integer
    "claims_that_held":    None,   # integer
    "anything_that_failed": "",    # what, or "nothing"
    "in_sample_or_out":    "",     # are the summary.csv figures in-sample or out-of-sample?
                                   # This one decides how much they are worth. Say which and why.
}
_m = [k for k, v in PASS1.items() if v is None or (isinstance(v, str) and not v.strip())]
print("Pass 1 complete." if not _m else "Pass 1 still to fill: " + ", ".join(_m))


---
# Pass 2 — find what it never tested

Everything above held. That is the point at which auditors stop, and it is the
point at which this lab starts.

The conclusion the cover note leads with — *at 36 months both optimised
portfolios lose to equal weighting; at 120 months both beat it* — comes from
`backtest.csv`. Print it.


In [ ]:
bt = pd.read_csv("lab06_backtest.csv")
print(bt.to_string(index=False))


Study that table before reading on.

**A hint, because the session is fifty minutes and this is the whole lab.** One
of the three portfolios in that table is a *control*. It is there precisely
because it does not do the thing being tested. Identify it, work out what its
numbers must look like if the comparison is sound, and then look at what they
actually are.

If you find it, you have found the defect. If you do not, run the next cell.


In [ ]:
# Run this only after you have tried. It narrows the search; it does not answer.
print(bt.pivot(index="portfolio", columns="window_m",
               values=["oos_months", "oos_sharpe"]).to_string())
print("\nEqual weighting holds the same 1/49 in every industry, every month.")
print("It estimates no means, no covariances, and no parameters of any kind.")
print("Ask yourself what its Sharpe ratio can possibly depend on.")


In [ ]:
PASS2 = {
    "the_control":            "",   # which portfolio, and why it is the control
    "what_it_should_do":      "",   # what its numbers must look like if the comparison is sound
    "what_it_actually_does":  "",   # with the numbers
    "what_that_proves":       "",   # state the defect in one sentence
}
_m = [k for k, v in PASS2.items() if not str(v).strip()]
print("Pass 2 complete." if not _m else "Pass 2 still to fill: " + ", ".join(_m))


---
# Pass 3 — re-run it properly

Having said what is wrong, fix it and measure the difference. A defect without a
magnitude is a complaint.

`walk_forward` below is the delivered backtest with **one parameter added**: the
month the out-of-sample period starts. That is the entire repair, which is worth
noticing — the defect is not a bug, it is a comparison that was never held fixed.

**This cell takes about a minute.**


In [ ]:
import time

def walk_forward(window, start=None, both=True):
    """Estimate on the trailing `window` months, hold one month, repeat.

    start : first out-of-sample month index. Leave as None to reproduce the
            delivered behaviour (each window starts as soon as it can).
            Set it to a fixed number to compare windows over ONE period.
    """
    R_, rf_ = data["R"], data["rf"]
    T = R_.shape[0]
    t0 = max(window, start or window)
    rows = []
    for t in range(t0, T):
        Rw = R_[t - window:t]
        mu = Rw.mean(axis=0)
        S, _ = OPT.cov_ledoit_wolf(Rw)
        rfm = float(np.nanmean(rf_[t - window:t]))
        r_mv = R_[t] @ OPT.min_variance(mu, S, True)
        r_tn = R_[t] @ OPT.tangency(mu, S, rfm, True) if both else np.nan
        rows.append((r_mv, r_tn, R_[t].mean(), rf_[t]))
    a = np.asarray(rows)
    rf_ann = a[:, 3].mean() * 12
    out = {"window": window, "oos_months": len(a), "first_oos": data["dates"][t0]}
    for i, nm in enumerate(("min_variance", "tangency", "equal_weight")):
        ann = a[:, i].mean() * 12
        vol = a[:, i].std(ddof=1) * np.sqrt(12)
        out[nm] = (ann - rf_ann) / vol if vol > 0 else np.nan
    return out

t = time.time()
delivered = [walk_forward(36), walk_forward(120)]
matched   = [walk_forward(36, start=120), walk_forward(120, start=120)]
print(f"({time.time() - t:.0f}s)\n")

print("AS DELIVERED — each window scored over its own period")
print(pd.DataFrame(delivered).to_string(index=False))
print("\nMATCHED — both windows scored over the same months")
print(pd.DataFrame(matched).to_string(index=False))


Now decompose it. The gap the cover note reports is the sum of two things: the
effect of the estimation window, which is what it claims to measure, and the
effect of the sample period, which it does not mention.


In [ ]:
rows = []
for p in ("min_variance", "tangency"):
    shown   = delivered[0][p]      # 36m, its own period
    matched36 = matched[0][p]      # 36m, matched period
    long_   = matched[1][p]        # 120m
    rows.append({"portfolio": p, "as_shown_36m": round(shown, 3),
                 "matched_36m": round(matched36, 3), "at_120m": round(long_, 3),
                 "period_effect": round(matched36 - shown, 3),
                 "window_effect": round(long_ - matched36, 3),
                 "pct_of_gap_that_is_period":
                     round(100 * (matched36 - shown) /
                           ((matched36 - shown) + (long_ - matched36)))})
print(pd.DataFrame(rows).to_string(index=False))

print(f"\nequal-weight, matched:  36m {matched[0]['equal_weight']:.3f}   "
      f"120m {matched[1]['equal_weight']:.3f}")
print("If those two are now identical, the comparison is finally controlled.")


---
# Pass 4 — the verdict

You have found a defect and measured it. Now the judgement, which is the part
that is actually difficult and the part a firm pays you for.

Three verdicts are available and they are not the same:

- **Wrong** — the conclusion does not survive correction.
- **Overstated** — the conclusion survives, but not at the size claimed.
- **Unsupported** — the conclusion may well be true; this evidence does not
  establish it.

Pick one, defend it with your numbers, and say what you would tell the person
who sent you the file.


In [ ]:
VERDICT = {
    "verdict":            "",     # wrong | overstated | unsupported
    "which_conclusion":   "",     # quote the claim you are ruling on
    "survives_correction": None,  # True / False
    "size_as_claimed":    None,   # the Sharpe gap the cover note implies
    "size_when_matched":  None,   # the Sharpe gap after your correction
    "what_you_would_say": "",     # one or two sentences, to the sender
}
_m = [k for k, v in VERDICT.items()
      if v is None or (isinstance(v, str) and not v.strip())]
print("Complete." if not _m else "Still to fill in: " + ", ".join(_m))


---
## Two remedies, and which one was actually load-bearing

Re-read what the specimen said it did: *"I handled it the standard way — **long-only
plus Ledoit-Wolf shrinkage**."*

Two remedies, applied together, and it never tested which one mattered. Neither
was asked for. The request said nothing about constraints, nothing about
estimators. Both were the system's own choices, and one of them is holding the
whole thing up.

Shrinkage replaces the sample covariance with a blend:

$$\Sigma(k) \;=\; k \cdot \bar{\lambda} I \;+\; (1-k)\, S$$

where $\bar\lambda$ is the average variance and $k \in [0,1]$. At $k=0$ you have
the singular sample matrix; at $k=1$ every correlation has been discarded.

Fill in the sweep and read the last column: **realised risk divided by the risk
the model predicted.** That ratio is what "error-maximisation machine" means.


In [ ]:
def shrink(S, k):
    lam = np.trace(S) / S.shape[0]
    return k * lam * np.eye(S.shape[0]) + (1.0 - k) * S

EST, OOS = data["R"][-72:-36], data["R"][-36:]   # fit on one 36m block, test on the next
mu_e = EST.mean(axis=0)
S_e = np.cov(EST, rowvar=False)

KS = []          # <-- fill this in: the values of k to try, from 0 to 1.
                 #     Include 0, include the specimen's 0.1605, and include 1.

def sweep(long_only):
    rows = []
    for k in KS:
        Sk = shrink(S_e, k)
        w = OPT.min_variance(mu_e, Sk, long_only)
        pred = np.sqrt(max(w @ Sk @ w, 0)) * np.sqrt(12) * 100
        real = (OOS @ w).std(ddof=1) * np.sqrt(12) * 100
        rows.append({"k": k, "cond": f"{np.linalg.cond(Sk):.3g}",
                     "predicted_vol_%": round(pred, 2),
                     "realised_vol_%": round(real, 2),
                     "realised / predicted": round(real / pred, 2) if pred > 1e-9 else np.inf,
                     "largest_weight": round(np.abs(w).max(), 2)})
    return pd.DataFrame(rows)

if not KS:
    print("Fill in KS above and run this cell.")
else:
    print("LONG-ONLY — the constraint the system chose, unasked\n")
    print(sweep(True).to_string(index=False))


Under long-only that table is unremarkable. The singular matrix at $k=0$ causes
no visible damage at all.

**So run it again with the constraint removed.** Nothing else changes — same
data, same estimator, same window, same singular covariance.


In [ ]:
if not KS:
    print("Fill in KS above first.")
else:
    print("SHORTING ALLOWED — same everything, one constraint removed\n")
    print(sweep(False).to_string(index=False))


In [ ]:
SHRINKAGE = {
    "ratio_at_k0_long_only":     None,  # realised / predicted, k = 0, long-only
    "ratio_at_k0_shorting":      None,  # realised / predicted, k = 0, shorting allowed
    "which_remedy_was_holding_it_up": "",   # long-only, or shrinkage? Say how you know.
    "was_that_remedy_requested": "",    # look at lab06_request.txt again
    "cost_of_over_shrinking":    "",    # k = 1 is bad too. Say why, in one sentence.
}
_m = [k for k, v in SHRINKAGE.items()
      if v is None or (isinstance(v, str) and not v.strip())]
print("Complete." if not _m else "Still to fill in: " + ", ".join(_m))


---
## Now ask the model

Give the backtest methodology to a model and ask whether the comparison supports
its conclusion. You have already done the work, so you can mark the answer.

**Then check what it says against your own pass 3.** If it agrees with you,
that is not confirmation — you can both be wrong the same way. Ask instead
whether it identified the control, because that is the step that settles it.


In [ ]:
PROMPT = """A portfolio research script reports the following walk-forward
out-of-sample results, and concludes from them that a 120-month estimation
window is materially better than a 36-month one.

""" + bt.to_string(index=False) + """

The three portfolios are: minimum variance and tangency, both re-estimated every
month from the trailing window; and equal weighting, which holds 1/49 in each of
49 industries every month and estimates nothing.

Does this table support the conclusion drawn from it? Answer the specific
question of whether the comparison isolates the effect of the estimation window,
and if it does not, say what else is varying and how you can tell from the table
itself."""

review = fina4030a.complete(PROMPT, temperature=0.0, max_tokens=900)
print(review)


In [ ]:
MODEL_REVIEW = {
    "did_it_find_the_confound":   None,  # True / False
    "did_it_use_the_control_row": None,  # True / False -- the step that settles it
    "anything_it_added":          "",    # or "nothing"
    "anything_it_got_wrong":      "",    # or "nothing" -- check before answering
}
_m = [k for k, v in MODEL_REVIEW.items()
      if v is None or (isinstance(v, str) and not v.strip())]
print("Complete." if not _m else "Still to fill in: " + ", ".join(_m))


---
## Findings


In [ ]:
FINDINGS = {
    "what_the_specimen_got_right": "",  # be specific; this is most of it
    "the_defect":                  "",  # one sentence
    "how_you_found_it":            "",  # the step that actually located it
    "size_of_the_correction":      "",  # with numbers
    "would_review_have_caught_it": "",  # would a partner reading the output page?
    "the_check_you_would_add":     "",  # one check this script should carry
    "confidence":                  None,  # 1-5
}
_m = [k for k, v in FINDINGS.items()
      if v is None or (isinstance(v, str) and not v.strip())]
print("Complete." if not _m else "Still to fill in: " + ", ".join(_m))

# Calibration log, checked here too — the "after" fields are easy to forget.
if "CALIBRATION" not in globals():
    print("Calibration log: run the calibration cell above before you submit.")
else:
    _cal = [k for k, v in CALIBRATION.items()
            if v is None or (isinstance(v, str) and not v.strip())]
    print("Calibration log complete."
          if not _cal else "Calibration log still to fill in: " + ", ".join(_cal))


In [ ]:
n_months = data["R"].shape[0]
print(fina4030a.appendix(
    student=f"{NAME} ({STUDENT_ID})",
    verification=FINDINGS.get("the_defect", ""),
    residual_risk=FINDINGS.get("would_review_have_caught_it", ""),
    reproducibility=(
        f"Audit of lab06_optimiser.py against lab06_returns.json "
        f"({n_months} months x {N} industries, {data['dates'][0]}-{data['dates'][-1]}). "
        f"Walk-forward re-run with matched out-of-sample start. "
        f"Shrinkage swept over {len(KS)} values of k."),
))

fina4030a.save_transcript("lab06_transcript.json")


---

## What to submit

1. This notebook with outputs intact.
2. `lab06_transcript.json`.
3. The appendix printed above.

## Two things to carry forward

**The first is about comparisons.** The specimen was good. Its arithmetic was
right, its diagnostics were honest, it volunteered bad news about its own
results, and its implementation of a non-trivial estimator matched the reference
library to within a rounding error of zero. Every check in pass 1 came back
clean. And then it drew a conclusion from a comparison that varied two things at
once.

No amount of care in the parts protects you from that, because it is not a
failure of care. It is a failure of *design*: the experiment could not have
distinguished the explanation offered from the obvious alternative, and executing
it perfectly would not have helped. The tell was sitting in its own results
table — a control that cannot move, and moved.

> When something is compared against something else, ask what was held fixed.
> Then find the row that proves it.

**The second is about the thing that did not break.** The nightmare this session
was designed around — a portfolio the model believes is riskless, that is not —
is real and you produced it in the shrinkage cell: predicted volatility of zero
against realised volatility in the thirties. It did not appear in the delivered
optimiser because of a **long-only constraint that nobody asked for**. The
request specified no constraints at all. The system chose one, described it in
passing alongside the estimator, and never tested which of the two was doing the
work.

It was the constraint. Change that one line and the same code, the same data and
the same estimator produce the catastrophe.

> The safety of a system is often carried by a choice nobody recorded as a
> safety decision. Find out which of your assumptions is load-bearing before
> someone relaxes it for a good reason.

---

### Next

**Class 7** is macro and rates, and a backtest that reports a result too good to
be true. Today's comparison was *confounded* — two things varied where one was
claimed. Next week's is *contaminated* — information reaches the test that would
not have been available at the time. They are cousins, they are not the same, and
telling them apart is most of what a quantitative reviewer does.
